In [ ]:
from google.colab import drive
drive.mount('/content/drive')
model_path = '/content/drive/MyDrive/model-weights'

Mounted at /content/drive


In [ ]:
!rm -r XION.tar

rm: cannot remove 'XION.tar': No such file or directory


In [ ]:
from huggingface_hub import hf_hub_download

# Get Hugging Face token from user input
repo_id = "AquaCoder0010/XION-Malware-Benign-dataset"
local_dir = "./"

file_path = hf_hub_download(
    repo_id=repo_id,               # Repository name
    filename="XION.tar",           # File to download
    repo_type="dataset",           # "model", "dataset", or "space"
    local_dir_use_symlinks=False,   # Don't use symlinks, actual copy
    local_dir=local_dir
)

print(f"All files downloaded to {local_dir}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


XION.tar:   0%|          | 0.00/41.7G [00:00<?, ?B/s]

All files downloaded to ./


In [ ]:
%%writefile punch_extract.py
import os, sys, ctypes, ctypes.util

FALLOC_FL_KEEP_SIZE  = 0x01
FALLOC_FL_PUNCH_HOLE = 0x02

libc = ctypes.CDLL(ctypes.util.find_library('c'), use_errno=True)

def punch(fd, offset, length):
    mode = FALLOC_FL_PUNCH_HOLE | FALLOC_FL_KEEP_SIZE
    if libc.fallocate(fd, mode, ctypes.c_longlong(offset), ctypes.c_longlong(length)) != 0:
        e = ctypes.get_errno()
        raise OSError(e, os.strerror(e))

def main(path, chunk=64 * 1024 * 1024):  # 64 MB
    fd = os.open(path, os.O_RDWR)
    out = sys.stdout.buffer
    punched = 0
    try:
        while True:
            data = os.read(fd, chunk)
            if not data:
                break
            out.write(data)
            pos = os.lseek(fd, 0, os.SEEK_CUR)
            length = pos - punched
            if length > 0:
                punch(fd, punched, length)
                punched = pos
        out.flush()
    finally:
        os.close(fd)

if __name__ == '__main__':
    main(sys.argv[1])

Writing punch_extract.py


In [ ]:
!python3 punch_extract.py XION.tar | tar -xf - -C ./

In [ ]:
!mkdir -p checkpoints

In [ ]:
import sys
import shutil
import os


value = None
if os.path.exists(f'{model_path}/last_trained.pth'):
  os.makedirs('checkpoints', exist_ok=True)
  shutil.copy(f'{model_path}/last_trained.pth', os.getcwd() + '/checkpoints')
  value = "script --train-dir XION/train --val-dir XION/val --max-seq-len 50000 --batch-size 64 --epochs 4 --checkpoint checkpoints/last_trained.pth"
else:
  value = "script --train-dir XION/train --val-dir XION/val --max-seq-len 50000 --batch-size 64 --epochs 4"
sys.argv = value.split()

# Dataset.py

In [ ]:
"""
Dataset classes for ByteFormer
Handles loading raw byte files (text files with binary byte strings).
"""

import os
import torch
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from typing import Optional, Tuple, List


class BytesTransform:
    """
    Transform that loads raw byte data from text files.

    The dataset contains text files where each line contains space-separated
    byte values (0-255), e.g., "23 156 89 255 0 34"
    """

    def __init__(
        self,
        max_length: Optional[int] = None,
        pad_value: int = -1
    ):
        """
        Args:
            max_length: Maximum sequence length (truncate if longer, pad if shorter)
            pad_value: Value used for padding shorter sequences
        """
        self.max_length = max_length
        self.pad_value = pad_value

    def __call__(self, filepath: Path) -> torch.Tensor:
        """
        Load byte data from a text file.

        Args:
            filepath: Path to the text file containing byte values

        Returns:
            Tensor of shape [seq_len] containing byte values as long
        """
        with open(filepath, 'rb') as f:
            content = f.read()

        # Parse space-separated byte values
        byte_values = list(content)
        byte_tensor = torch.tensor(byte_values, dtype=torch.long)

        # Handle max_length
        if self.max_length is not None:
            seq_len = byte_tensor.shape[0]

            if seq_len > self.max_length:
                # Truncate
                byte_tensor = byte_tensor[:self.max_length]
            elif seq_len < self.max_length:
                # Pad
                padding = torch.full((self.max_length - seq_len,), self.pad_value, dtype=torch.long)
                byte_tensor = torch.cat([byte_tensor, padding])

        return byte_tensor


class ByteFormerDataset(Dataset):
    """
    Dataset that loads raw byte sequences from text files.

    Expected directory structure:
        root/
            class_0/
                file1.txt
                file2.txt
            class_1/
                file3.txt
    """

    def __init__(
        self,
        root: str,
        transform: Optional[BytesTransform] = None,
        max_length: Optional[int] = None,
        extensions: Tuple[str, ...] = (".rtc",)
    ):
        """
        Args:
            root: Root directory containing class subdirectories
            transform: Optional transform to apply to byte sequences
            max_length: Maximum sequence length
            extensions: Valid file extensions
        """
        self.root = Path(root)
        self.transform = transform or BytesTransform(max_length=max_length)

        # Build list of (file_path, class_idx) tuples
        self.samples: List[Tuple[Path, int]] = []
        self.class_to_idx = {}

        for class_name in sorted(os.listdir(root)):
            class_dir = self.root / class_name
            if not class_dir.is_dir():
                continue

            class_idx = len(self.class_to_idx)
            self.class_to_idx[class_name] = class_idx

            for file_path in class_dir.iterdir():
                if file_path.suffix.lower() in extensions:
                    self.samples.append((file_path, class_idx))

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, int]:
        filepath, label = self.samples[idx]

        # Load and transform byte sequence
        bytes_tensor = self.transform(filepath)

        return bytes_tensor, label

    def get_class_counts(self) -> dict:
        """Get count of samples per class."""
        counts = {}
        for _, label in self.samples:
            counts[label] = counts.get(label, 0) + 1
        return counts


class CollateFn:
    """Collate function to handle variable-length byte sequences."""

    def __init__(self, pad_value: int = -1):
        """
        Args:
            pad_value: Value used for padding shorter sequences
        """
        self.pad_value = pad_value

    def __call__(self, batch: List[Tuple[torch.Tensor, int]]) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Collate batch of (bytes, label) tuples.

        Returns:
            padded_bytes: [batch_size, max_seq_len]
            labels: [batch_size]
        """
        bytes_list, labels = zip(*batch)

        # Get max sequence length in batch
        max_seq_len = max(b.shape[0] for b in bytes_list)

        # Pad sequences
        padded = torch.full((len(bytes_list), max_seq_len), self.pad_value, dtype=torch.long)

        for i, bytes_tensor in enumerate(bytes_list):
            seq_len = bytes_tensor.shape[0]
            padded[i, :seq_len] = bytes_tensor

        labels = torch.tensor(labels, dtype=torch.long)

        return padded, labels


def create_dataloader(
    dataset: Dataset,
    batch_size: int = 32,
    shuffle: bool = True,
    num_workers: int = 4,
    **kwargs
) -> DataLoader:
    """
    Create a DataLoader with custom collate function.

    Args:
        dataset: Dataset to load from
        batch_size: Number of samples per batch
        shuffle: Whether to shuffle data
        num_workers: Number of worker processes
        **kwargs: Additional arguments for CollateFn

    Returns:
        DataLoader instance
    """
    collate_fn = CollateFn(**kwargs)
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        collate_fn=collate_fn
    )


# model.py


In [ ]:
"""
ByteFormer Model Implementation with accurate WindowedAttention from corenet.
"""

import argparse
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor
from typing import Tuple, Optional


class ByteFormerConfig:
    """Configuration class for ByteFormer model sizes."""

    MODES = {
        "tiny": {"embed_dim": 192, "n_layers": 12, "n_heads": 3, "ffn_dim": 768},
        "small": {"embed_dim": 384, "n_layers": 12, "n_heads": 6, "ffn_dim": 1536},
        "base": {"embed_dim": 768, "n_layers": 12, "n_heads": 12, "ffn_dim": 3072},
    }


def window_partition(t: torch.Tensor, window_size: int) -> torch.Tensor:
    """
    Partition tensor into chunks of size window_size.

    Args:
        t: Tensor of shape [batch_size, sequence_length, embed_dim]
        window_size: The desired window size

    Returns:
        Tensor of shape [batch_size * num_windows, window_size, embed_dim]
    """
    B, N, C = t.shape
    if not N % window_size == 0:
        raise ValueError(f"sequence length {N} must be divisible by window size {window_size}")
    t = t.reshape(B * N // window_size, window_size, C)
    return t


def window_partition_reverse(t: torch.Tensor, B: int, num_windows: int, C: int) -> torch.Tensor:
    """Reverse the window partition operation."""
    t = t.reshape(B, num_windows * t.shape[1], C)
    return t


def get_windows_shift_mask(N: int, window_size: int, window_shift: int, device: torch.device) -> torch.Tensor:
    """
    Get mask for shifted window attention.

    Args:
        N: Sequence length
        window_size: Window size
        window_shift: Shift amount
        device: Device for tensor

    Returns:
        Mask tensor of shape [N // window_size, window_size, window_size]
    """
    ret = torch.zeros(N // window_size, window_size, window_size, device=device)
    ret[-1].fill_(float("-inf"))
    ret[-1, :window_size - window_shift, :window_size - window_shift] = 0
    ret[-1, -window_shift:, -window_shift:] = 0
    return ret


def pad_x_and_mask(x: torch.Tensor, key_padding_mask: torch.Tensor, window_size: int):
    """Pad input to be divisible by window size."""
    B, N = key_padding_mask.shape
    pad_len = (window_size - N % window_size) % window_size

    if pad_len > 0:
        x = torch.cat([x, torch.zeros(B, pad_len, x.shape[2], device=x.device)], dim=1)
        key_padding_mask = torch.cat([key_padding_mask, torch.full((B, pad_len), float("-inf"), device=key_padding_mask.device)], dim=1)

    return x, key_padding_mask


def window_x_and_key_padding_mask(
    x: torch.Tensor, key_padding_mask: torch.Tensor, window_size: int, window_shift: int
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Perform windowing on input and mask for windowed attention."""
    B, N = key_padding_mask.shape
    assert x.shape[:2] == (B, N)

    x, key_padding_mask = pad_x_and_mask(x, key_padding_mask, window_size)

    if window_shift > 0:
        x = torch.roll(x, shifts=(-window_shift), dims=1)
        key_padding_mask = torch.roll(key_padding_mask, shifts=(-window_shift), dims=1)

    x_windows = window_partition(x, window_size)
    token_mask_windows = key_padding_mask.reshape(B * x.shape[1] // window_size, window_size)
    window_mask = get_windows_shift_mask(x.shape[1], window_size, window_shift, x_windows.device).expand(B, -1, -1, -1)
    window_mask = window_mask.reshape(window_mask.shape[0] * window_mask.shape[1], window_mask.shape[2], window_mask.shape[3])

    return x_windows, token_mask_windows, window_mask


def unwindow_x(x_windows: torch.Tensor, B: int, N: int, C: int, window_shift: int):
    """Reverse the windowing operation."""
    num_windows = x_windows.shape[0] // B
    x = window_partition_reverse(x_windows, B, num_windows, C)

    if window_shift > 0:
        x = torch.roll(x, shifts=window_shift, dims=1)
    x = x[:, :N]

    return x


class MultiHeadAttention(nn.Module):
    """
    Multi-head attention from corenet implementation.
    """

    def __init__(self, embed_dim: int, num_heads: int, attn_dropout: float = 0.0):
        super().__init__()
        if embed_dim % num_heads != 0:
            raise ValueError(f"embed_dim {embed_dim} must be divisible by num_heads {num_heads}")

        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.scaling = self.head_dim ** -0.5

        self.qkv_proj = nn.Linear(embed_dim, 3 * embed_dim)
        self.out_proj = nn.Linear(embed_dim, embed_dim)
        self.attn_dropout = nn.Dropout(attn_dropout)
        self.softmax = nn.Softmax(dim=-1)

    def forward(
        self, x: Tensor,
        key_padding_mask: Optional[Tensor] = None,
        attn_mask: Optional[Tensor] = None
    ) -> Tensor:
        """
        Forward pass for multi-head attention.

        Args:
            x: Input tensor [B, N, C]
            key_padding_mask: Padding mask [B, N]
            attn_mask: Attention mask

        Returns:
            Output tensor [B, N, C]
        """
        B, S_len, _ = x.shape

        # Compute QKV
        qkv = self.qkv_proj(x).reshape(B, S_len, 3, self.num_heads, self.head_dim)
        qkv = qkv.transpose(1, 3).contiguous()
        query, key, value = qkv[:, :, 0], qkv[:, :, 1], qkv[:, :, 2]

        query = query * self.scaling
        key = key.transpose(-2, -1)

        # QK^T
        attn = torch.matmul(query, key)

        batch_size, num_heads, num_src_tokens, num_tgt_tokens = attn.shape

        if attn_mask is not None:
            attn_mask = attn_mask.unsqueeze(1)
            attn = attn + attn_mask

        if key_padding_mask is not None:
            attn = attn.masked_fill(
                key_padding_mask.unsqueeze(1).unsqueeze(2).bool(),
                float("-inf")
            )

        attn = self.softmax(attn.float()).to(attn.dtype)
        attn = self.attn_dropout(attn)

        # Weighted sum
        out = torch.matmul(attn, value)
        out = out.transpose(1, 2).reshape(B, S_len, -1)
        out = self.out_proj(out)

        return out


class TransformerEncoderLayer(nn.Module):
    """
    Single transformer encoder layer with pre-norm.
    """

    def __init__(self, embed_dim: int, num_heads: int, ffn_dim: int, dropout: float = 0.0):
        super().__init__()

        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = MultiHeadAttention(embed_dim, num_heads)
        self.dropout1 = nn.Dropout(dropout)

        self.norm2 = nn.LayerNorm(embed_dim)
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, ffn_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ffn_dim, embed_dim),
            nn.Dropout(dropout)
        )

    def forward(self, x: Tensor, key_padding_mask: Optional[Tensor] = None, attn_mask: Optional[Tensor] = None) -> Tensor:
        # Multi-head attention with pre-norm
        res = x
        x = self.norm1(x)
        x = self.attn(x, key_padding_mask, attn_mask)
        x = res + self.dropout1(x)

        # FFN with pre-norm
        x = x + self.ffn(self.norm2(x))

        return x


class WindowedTransformerEncoder(nn.Module):
    """
    Transformer encoder with windowed (shifted window) attention.
    This is the accurate implementation from corenet.
    """

    def __init__(
        self,
        embed_dim: int,
        num_heads: int,
        ffn_dim: int,
        window_size: int,
        window_shift: int,
        dropout: float = 0.0
    ):
        super().__init__()

        self.encoder_layer = TransformerEncoderLayer(embed_dim, num_heads, ffn_dim, dropout)
        self.window_size = window_size
        self.window_shift = window_shift

    def forward(
        self,
        x: Tensor,
        key_padding_mask: Optional[Tensor] = None,
        attn_mask: Optional[Tensor] = None
    ) -> Tensor:
        """
        Forward pass with windowing.

        Args:
            x: Input tensor [B, N, C]
            key_padding_mask: Padding mask [B, N]
            attn_mask: Additional attention mask

        Returns:
            Output tensor [B, N, C]
        """
        B, N, C = x.shape

        # Handle None key_padding_mask
        if key_padding_mask is None:
            key_padding_mask = torch.zeros(B, N, dtype=torch.float, device=x.device)

        # Window the input
        x_windows, windowed_key_padding_mask, windows_mask = window_x_and_key_padding_mask(
            x, key_padding_mask, self.window_size, self.window_shift
        )

        total_mask = windowed_key_padding_mask.unsqueeze(1) + windows_mask

        if attn_mask is not None:
            total_mask += attn_mask

        # Handle fully masked windows to avoid NaN
        fully_masked_windows = total_mask.max(dim=-1).values == float("-inf")
        total_mask[fully_masked_windows] = 0

        # Pass through transformer layer
        x_windows = self.encoder_layer(x_windows, attn_mask=total_mask)


        # Unwindow the output
        x = unwindow_x(x_windows, B, N, C, self.window_shift)

        return x


class TokenReduction(nn.Module):
    """Conv1d layer to reduce sequence length before transformer."""

    def __init__(self, embed_dim: int, kernel_size: int):
        super().__init__()
        self.conv = nn.Conv1d(
            embed_dim, embed_dim,
            kernel_size=kernel_size,
            stride=kernel_size // 2,
            bias=False
        )
        # 90 %


    def forward(self, x: Tensor) -> Tensor:
        # 99 % reduction in size.
        return self.conv(self.conv(x.permute(0, 2, 1))).permute(0, 2, 1)


class PositionalEmbedding(nn.Module):
    """Learnable positional embeddings for byte sequences."""

    def __init__(self, num_embeddings: int, embedding_dim: int):
        super().__init__()
        self.embedding = nn.Embedding(num_embeddings, embedding_dim)

    def forward(self, seq_len: int) -> Tensor:
        positions = torch.arange(seq_len, device=self.embedding.weight.device)
        return self.embedding(positions)


class ByteFormer(nn.Module):
    """
    ByteFormer: Vision Transformer operating on raw image bytes.
    With accurate WindowedTransformerEncoder from corenet.
    """

    def __init__(
        self,
        vocab_size: int = 257,
        embed_dim: int = 192,
        n_layers: int = 12,
        n_heads: int = 3,
        ffn_dim: int = 768,
        conv_kernel_size: int = 16,
        max_num_tokens: int = 50000,
        num_classes: int = 1000,
        window_size: int = 128,
        window_shift: int = 64,
        dropout: float = 0.0
    ):
        super().__init__()

        self.embed_dim = embed_dim
        self.vocab_size = vocab_size

        # Byte embedding layer
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=256)

        # Token reduction convolution
        self.token_reduction = TokenReduction(embed_dim, conv_kernel_size) if conv_kernel_size > 0 else None

        # Positional embeddings
        self.pos_embed = PositionalEmbedding(max_num_tokens, embed_dim)

        # Transformer backbone with windowed attention
        self.transformer = nn.ModuleList([
            WindowedTransformerEncoder(
                embed_dim=embed_dim,
                num_heads=n_heads,
                ffn_dim=ffn_dim,
                window_size=window_size,
                window_shift=window_shift if i % 2 == 1 else 0,  # Alternate shift
                dropout=dropout
            )
            for i in range(n_layers)
        ])

        self.norm = nn.LayerNorm(embed_dim)

        # Classification head
        self.classifier = nn.Linear(embed_dim, num_classes)

        self._init_weights()

    def _init_weights(self):
        """Initialize embedding weights."""
        nn.init.trunc_normal_(self.embedding.weight[:-1], std=math.sqrt(1.0 / self.embed_dim))

    def get_backbone_inputs(self, x: Tensor) -> Tuple[Tensor, Tensor]:
        # 1. Create initial mask based on indices before they are embedded
        mask = torch.zeros_like(x, dtype=torch.float)
        mask[x == 256] = float("-inf")

        # 2. Embed the bytes
        x_indices = x.clone()
        x_embeds = self.embedding(x_indices)

        # 3. Apply Token Reduction (Convolution)
        if self.token_reduction is not None:
            # Reduce the data
            x_embeds = self.token_reduction(x_embeds)

            is_pad = (mask == float("-inf")).float().unsqueeze(1) # [B, 1, N]

            # FIRST POOL
            is_pad_pooled = F.max_pool1d(
                is_pad,
                kernel_size=self.token_reduction.conv.kernel_size[0],
                stride=self.token_reduction.conv.stride[0],
                padding=self.token_reduction.conv.padding[0]
            )

            # SECOND POOL (Added to match the double-conv in TokenReduction)
            is_pad_pooled = F.max_pool1d(
                is_pad_pooled,
                kernel_size=self.token_reduction.conv.kernel_size[0],
                stride=self.token_reduction.conv.stride[0],
                padding=self.token_reduction.conv.padding[0]
            ).squeeze(1)

            # Overwrite the original mask variable
            mask = torch.zeros_like(is_pad_pooled)
            mask[is_pad_pooled > 0] = float("-inf")

        # 4. Add positional embeddings to the REDUCED sequence
        seq_len = x_embeds.shape[1]
        x_embeds = x_embeds + self.pos_embed(seq_len)

        return x_embeds, mask

    def forward(self, x: Tensor) -> Tensor:
        """
        Forward pass through ByteFormer.

        Args:
            x: Input tensor [B, N] containing byte values

        Returns:
            Logits [B, num_classes]
        """
        x, mask = self.get_backbone_inputs(x)

        # Pass through transformer layers
        for block in self.transformer:
            x = block(x, mask)

        x = self.norm(x)

        # Mean pooling
        mask_expanded = mask.unsqueeze(-1)
        x = x.masked_fill(mask_expanded == float("-inf"), 0)
        x = x.sum(dim=1) / (mask != float("-inf")).sum(dim=1).unsqueeze(-1).clamp(min=1)

        return self.classifier(x)

def create_byteformer(mode: str = "tiny", num_classes: int = 1000, **kwargs) -> ByteFormer:
    """
    Factory function to create a ByteFormer model.
    """
    # 1. Get a copy of the base configuration for the selected mode
    config = ByteFormerConfig.MODES[mode].copy()

    # 2. Update kwargs with the mode config
    # (This safely overwrites any conflicting keys like 'embed_dim' in kwargs)
    kwargs.update(config)

    # 3. Unpack the merged dictionary into the model
    return ByteFormer(
        num_classes=num_classes,
        **kwargs
    )


# Train.py

In [ ]:
"""
Training script for ByteFormer model.
"""

import os
import argparse
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.cuda.amp import autocast, GradScaler
from pathlib import Path
from tqdm import tqdm

def train_one_epoch(
    model: nn.Module,
    dataloader: DataLoader,
    criterion: nn.Module,
    optimizer: optim.Optimizer,
    device: torch.device,
    scaler: GradScaler,
    epoch: int
) -> float:
    """
    Train for one epoch.

    Args:
        model: ByteFormer model
        dataloader: Training data loader
        criterion: Loss function
        optimizer: Optimizer
        device: Computation device
        scaler: Gradient scaler for mixed precision
        epoch: Current epoch number

    Returns:
        Average training loss
    """
    model.train()
    total_loss = 0.0

    pbar = tqdm(dataloader, desc=f"Epoch {epoch}")
    for batch_idx, (inputs, targets) in enumerate(pbar):
        inputs = inputs.to(device)
        targets = targets.to(device)

        optimizer.zero_grad()

        # Mixed precision training
        with autocast():
            outputs = model(inputs)
            loss = criterion(outputs, targets)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        pbar.set_postfix({"loss": loss.item()})

    return total_loss / len(dataloader)


def evaluate(
    model: nn.Module,
    dataloader: DataLoader,
    criterion: nn.Module,
    device: torch.device
) -> dict:
    """
    Evaluate model on validation set.

    Args:
        model: ByteFormer model
        dataloader: Validation data loader
        criterion: Loss function
        device: Computation device

    Returns:
        Dictionary with evaluation metrics
    """
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, targets in dataloader:
            inputs = inputs.to(device)
            targets = targets.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, targets)

            total_loss += loss.item()
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()

    return {
        "loss": total_loss / len(dataloader),
        "accuracy": 100.0 * correct / total
    }


def save_checkpoint(
    model: nn.Module,
    optimizer: optim.Optimizer,
    epoch: int,
    metrics: dict,
    save_path: str
):
    """Save model checkpoint."""
    checkpoint = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "metrics": metrics
    }
    torch.save(checkpoint, save_path)
    print(f"Checkpoint saved to {save_path}")


def test(
    model: nn.Module,
    dataloader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
    num_classes: int
) -> dict:
    """
    Evaluate model on test set with detailed metrics.

    Args:
        model: ByteFormer model
        dataloader: Test data loader
        criterion: Loss function
        device: Computation device
        num_classes: Number of classes

    Returns:
        Dictionary with evaluation metrics including per-class accuracy
    """
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    # Per-class tracking
    class_correct = [0] * num_classes
    class_total = [0] * num_classes

    # Top-k tracking
    top3_correct = 0
    top5_correct = 0

    all_preds = []
    all_targets = []
    all_probs = []

    with torch.no_grad():
        for inputs, targets in dataloader:
            inputs = inputs.to(device)
            targets = targets.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, targets)

            total_loss += loss.item()

            # Get probabilities for top-k
            probs = torch.softmax(outputs, dim=1)

            # Top-1 accuracy
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()

            # Per-class accuracy
            for i in range(targets.size(0)):
                label = targets[i].item()
                class_total[label] += 1
                if predicted[i] == targets[i]:
                    class_correct[label] += 1

            all_preds.extend(predicted.cpu().numpy())
            all_targets.extend(targets.cpu().numpy())
            all_probs.append(probs.cpu())

    # Calculate metrics
    accuracy = 100.0 * correct / total
    avg_loss = total_loss / len(dataloader)

    # Per-class accuracy
    class_accuracy = {}
    for i in range(num_classes):
        if class_total[i] > 0:
            class_accuracy[i] = 100.0 * class_correct[i] / class_total[i]

    print("\n" + "=" * 50)
    print("TEST RESULTS")
    print("=" * 50)
    print(f"Total samples:     {total}")
    print(f"Average loss:     {avg_loss:.4f}")
    print(f"Top-1 Accuracy:   {accuracy:.2f}%")
    print("-" * 50)
    print("Per-class Accuracy:")
    for i, acc in class_accuracy.items():
        print(f"  Class {i}: {acc:.2f}% ({class_correct[i]}/{class_total[i]})")
    print("=" * 50 + "\n")

    return {
        "loss": avg_loss,
        "accuracy": accuracy,
        "class_accuracy": class_accuracy,
        "class_correct": class_correct,
        "class_total": class_total
    }


def main(args):
    """Main training function."""
    # Set device
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    model = create_byteformer(
        mode=args.mode,
        num_classes=2,
        vocab_size=args.vocab_size,
        embed_dim=args.embed_dim,
        n_layers=args.n_layers,
        n_heads=args.n_heads,
        ffn_dim=args.ffn_dim,
        conv_kernel_size=args.conv_kernel_size,
        max_num_tokens=args.max_seq_len,
        window_size=args.window_size
    ).to(device)


    if args.test_dir:
        # Create test dataset
        test_transform = BytesTransform(max_length=args.max_seq_len)

        test_dataset = ByteFormerDataset(
            root=args.test_dir,
            transform=test_transform
        )

        test_loader = create_dataloader(
            test_dataset,
            batch_size=args.batch_size,
            shuffle=False,
            num_workers=args.num_workers
        )

        print(f"Test samples: {len(test_dataset)}")

        criterion = nn.CrossEntropyLoss()
        test_metrics = test(model, test_loader, criterion, device, len(test_dataset.class_to_idx))
        return


    # Create transforms
    train_transform = BytesTransform(max_length=args.max_seq_len)

    # Create datasets
    train_dataset = ByteFormerDataset(
        root=args.train_dir,
        transform=train_transform
    )

    val_dataset = ByteFormerDataset(
        root=args.val_dir,
        transform=train_transform
    )

    print(f"Train samples: {len(train_dataset)}, Val samples: {len(val_dataset)}")

    # Create dataloaders
    train_loader = create_dataloader(
        train_dataset,
        batch_size=args.batch_size,
        shuffle=True,
        num_workers=args.num_workers
    )

    val_loader = create_dataloader(
        val_dataset,
        batch_size=args.batch_size,
        shuffle=False,
        num_workers=args.num_workers
    )
    optimizer = optim.AdamW(
        model.parameters(),
        lr=args.learning_rate,
        weight_decay=args.weight_decay
    )
    # Load checkpoint
    if args.checkpoint:
        print(f"Loading checkpoint from {args.checkpoint}")
        checkpoint = torch.load(args.checkpoint, map_location=device)
        model.load_state_dict(checkpoint["model_state_dict"])
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])


    # Loss and optimizer
    criterion = nn.CrossEntropyLoss(label_smoothing=args.label_smoothing)


    # Learning rate scheduler
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=args.epochs
    )
    # Mixed precision scaler
    scaler = GradScaler()
    # Training loop
    best_acc = 0.0
    os.makedirs(args.save_dir, exist_ok=True)

    for epoch in range(1, args.epochs + 1):
        train_loss = train_one_epoch(
            model, train_loader, criterion, optimizer, device, scaler, epoch
        )
        val_metrics = evaluate(model, val_loader, criterion, device)

        scheduler.step()

        print(f"Epoch {epoch}/{args.epochs}:")
        print(f"  Train Loss: {train_loss:.4f}")
        print(f"  Val Loss: {val_metrics['loss']:.4f}, Val Acc: {val_metrics['accuracy']:.2f}%")

        # Save best model
        if val_metrics["accuracy"] > best_acc:
            best_acc = val_metrics["accuracy"]
            save_checkpoint(
                model, optimizer, epoch, val_metrics,
                os.path.join(args.save_dir, "best_model.pth")
            )

        # Save periodic checkpoint
        if epoch % args.save_freq == 0:
            save_checkpoint(
                model, optimizer, epoch, val_metrics,
                os.path.join(args.save_dir, f"checkpoint_epoch_{epoch}.pth")
            )

    print(f"Training complete! Best accuracy: {best_acc:.2f}%")
    print(f"saving latest and best model to Google Drive")

    last_trained = sorted( [ value for value in os.listdir('checkpoints') if value[-4:] == '.pth' and value != 'best_model.pth'], key=lambda x: int(x.split('_')[-1].split('.')[0]) )[-1]
    shutil.copy(f'checkpoints/{last_trained}', f'{model_path}/{last_trained}')
    shutil.copy('checkpoints/best_model.pth', f'{model_path}/best_model.pth')


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="Train ByteFormer model")

    # Data arguments
    parser.add_argument("--train-dir", type=str, required=False, help="Training data directory")
    parser.add_argument("--val-dir", type=str, required=False, help="Validation data directory")
    parser.add_argument("--max-seq-len", type=int, default=100000, help="Maximum sequence length")

    # Model arguments
    parser.add_argument("--mode", type=str, default="tiny", choices=["tiny", "small", "base"])
    parser.add_argument("--vocab-size", type=int, default=257)
    parser.add_argument("--embed-dim", type=int, default=192)
    parser.add_argument("--n-layers", type=int, default=12)
    parser.add_argument("--n-heads", type=int, default=3)
    parser.add_argument("--ffn-dim", type=int, default=768)
    parser.add_argument("--conv-kernel-size", type=int, default=16)
    parser.add_argument("--window-size", type=int, default=128)

    # Training arguments
    parser.add_argument("--batch-size", type=int, default=32)
    parser.add_argument("--epochs", type=int, default=100)
    parser.add_argument("--learning-rate", type=float, default=1e-3)
    parser.add_argument("--weight-decay", type=float, default=0.05)
    parser.add_argument("--label-smoothing", type=float, default=0.1)
    parser.add_argument("--num-workers", type=int, default=4)

    # Save arguments
    parser.add_argument("--save-dir", type=str, default="checkpoints")
    parser.add_argument("--save-freq", type=int, default=1)

    # Test arguments
    parser.add_argument("--test-dir", type=str, default=None, help="Test data directory (for testing mode)")
    parser.add_argument("--checkpoint", type=str, default=None, help="Path to model checkpoint for testing")

    args = parser.parse_args()
    main(args)


Using device: cuda
Train samples: 48215, Val samples: 5994
Loading checkpoint from checkpoints/last_trained.pth


/tmp/ipykernel_12823/195130363.py:316: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
Epoch 1:   0%|          | 0/754 [00:00<?, ?it/s]/tmp/ipykernel_12823/195130363.py:50: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Epoch 1: 100%|██████████| 754/754 [53:01<00:00,  4.22s/it, loss=0.332]


Epoch 1/4:
  Train Loss: 0.3493
  Val Loss: 0.3471, Val Acc: 91.11%
Checkpoint saved to checkpoints/best_model.pth
Checkpoint saved to checkpoints/checkpoint_epoch_1.pth


Epoch 2: 100%|██████████| 754/754 [53:22<00:00,  4.25s/it, loss=0.393]


Epoch 2/4:
  Train Loss: 0.3408
  Val Loss: 0.3389, Val Acc: 91.52%
Checkpoint saved to checkpoints/best_model.pth
Checkpoint saved to checkpoints/checkpoint_epoch_2.pth


Epoch 3: 100%|██████████| 754/754 [53:28<00:00,  4.26s/it, loss=0.314]


Epoch 3/4:
  Train Loss: 0.3343
  Val Loss: 0.3344, Val Acc: 92.06%
Checkpoint saved to checkpoints/best_model.pth
Checkpoint saved to checkpoints/checkpoint_epoch_3.pth


Epoch 4: 100%|██████████| 754/754 [53:20<00:00,  4.25s/it, loss=0.243]


Epoch 4/4:
  Train Loss: 0.3266
  Val Loss: 0.3321, Val Acc: 92.08%
Checkpoint saved to checkpoints/best_model.pth
Checkpoint saved to checkpoints/checkpoint_epoch_4.pth
Training complete! Best accuracy: 92.08%
saving latest and best model to Google Drive


ValueError: invalid literal for int() with base 10: 'trained'